# From a dataset to a defensible learning task

**Case study.** The template uses chemically analyzed wine samples to demonstrate how a professional project begins. Before asking an algorithm to learn, we must establish what one row means, where the measurements came from, what can be known at use time, and which questions the data can actually support.

This report uses the project package for configuration, ingestion, provenance, and task definitions. Replace the example adapter with your team adapter; do not replace validation with assumptions made from a plot.

## Questions for this report

We will ask:

1. Did the configured adapter deliver the expected observational unit?
2. Are identifiers, types, missing values, and duplicates consistent with the contract?
3. What ranges and distributions characterize the chemistry measurements?
4. Are the declared regression, classification, and unsupervised tasks distinct and scientifically interpretable?

The answers are development evidence. They do not establish that the sample represents all wines or that chemistry causes cultivar identity.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from dsm_project.config import load_config
from dsm_project.data import load_project_dataset

config = load_config("configs/example.toml")
dataset = load_project_dataset(config)
frame = dataset.observations

print("Project:", config.project["name"])
print("Rows, columns:", frame.shape)
print("Observation unit:", dataset.provenance.observation_unit)
print("Source:", dataset.provenance.source)
display(frame.head())

## Audit before analysis

A preview is useful for orientation, but it cannot prove data quality. We inspect every column's type, missingness, uniqueness, and numerical range because these properties determine which transformations and failure tests belong in the package.

The identifier is a traceability key, not a feature. A unique integer ID may correlate with collection order and can create spectacular but meaningless predictions if it leaks into the feature matrix.

In [ ]:
audit = pd.DataFrame(
    {
        "dtype": frame.dtypes.astype(str),
        "missing": frame.isna().sum(),
        "unique": frame.nunique(dropna=False),
    }
)
audit["missing_fraction"] = audit["missing"] / len(frame)

assert frame[dataset.identifier].is_unique
assert frame[dataset.identifier].notna().all()
display(audit)

numeric_summary = frame.drop(columns=dataset.identifier).describe().T
display(numeric_summary[["count", "mean", "std", "min", "50%", "max"]])

## Distribution is more informative than one average

Location and scale matter for optimization, distance, margins, and regularization. Shape matters too: skew, outliers, multimodality, and bounded measurements can make two columns with similar means behave very differently. Histograms below are exploratory prompts, not automatic evidence that observations form natural groups.

In [ ]:
chemistry = list(dataset.tasks["classification"].feature_columns)
axes = frame[chemistry[:6]].hist(figsize=(12, 7), bins=18, color="#1f77b4", edgecolor="white")
for axis in axes.ravel():
    axis.set_ylabel("sample count")
plt.suptitle("Selected wine-chemistry distributions", y=1.02)
plt.tight_layout();

## Three tasks, three mathematical contracts

- **Regression** asks for a continuous estimate of alcohol concentration from the other measurements. Error retains the target's units.
- **Classification** asks which recorded cultivar label is associated with the chemistry vector. Class balance and error consequences matter.
- **Unsupervised learning** asks whether recurring chemistry profiles appear without using cultivar labels during fitting.

A column can be legitimate for one task and leakage for another. The package, rather than the notebook, owns these feature and target declarations so tests, scripts, and services use the same contract.

In [ ]:
for task_key, task in dataset.tasks.items():
    print(f"{task_key}: {task.name}")
    print("  kind:", task.kind)
    print("  feature count:", len(task.feature_columns))
    print("  target:", task.target_column)
    print("  metric:", task.metric)

class_counts = frame["cultivar"].value_counts().sort_index()
class_counts.plot.bar(color="#d95f02", rot=0)
plt.title("Recorded cultivar balance")
plt.xlabel("cultivar label")
plt.ylabel("sample count")
plt.tight_layout();

## Conclusion and student extension

The adapter delivers 178 uniquely identified, complete rows and declares separate tasks without including either the identifier or regression target in an invalid feature role. That is enough to begin controlled experiments, not enough to claim external validity.

For your project:

1. configure and load data through `dsm_project.data`;
2. complete the dataset card with source URL, license, retrieval date, observation unit, units, and dependence;
3. add schema and leakage tests before modeling;
4. replace generic plots with visuals tied to your domain question; and
5. write down what the available sample cannot represent.

Use the public-source catalog in `docs/PUBLIC_DATA_SOURCES.md`, but verify the license and record-level access terms of the specific dataset you select.